In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import json
import os
from datetime import datetime

# Dataset
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraîner le modèle
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy : {accuracy:.4f}")

# SAUVEGARDER LE MODÈLE
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/random_forest_v1.pkl")
print("✅ Modèle sauvegardé → models/random_forest_v1.pkl")

# SAUVEGARDER LES MÉTADONNÉES
metadata = {
    "model_name": "RandomForest",
    "version": "1.0",
    "date_trained": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "accuracy": accuracy,
    "n_estimators": 100,
    "n_features": 10,
    "n_samples_train": len(X_train),
    "n_samples_test": len(X_test)
}

with open("models/metadata_v1.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Métadonnées sauvegardées → models/metadata_v1.json")

# CHARGER LE MODÈLE
model_loaded = joblib.load("models/random_forest_v1.pkl")
accuracy_loaded = accuracy_score(y_test, model_loaded.predict(X_test))
print(f"\n✅ Modèle chargé avec succès")
print(f"Accuracy après chargement : {accuracy_loaded:.4f}")

# Afficher les métadonnées
with open("models/metadata_v1.json") as f:
    meta = json.load(f)
print(f"\n=== MÉTADONNÉES ===")
for k, v in meta.items():
    print(f"  {k:20s} : {v}")

Accuracy : 0.8800
✅ Modèle sauvegardé → models/random_forest_v1.pkl
✅ Métadonnées sauvegardées → models/metadata_v1.json

✅ Modèle chargé avec succès
Accuracy après chargement : 0.8800

=== MÉTADONNÉES ===
  model_name           : RandomForest
  version              : 1.0
  date_trained         : 2026-06-20 09:54
  accuracy             : 0.88
  n_estimators         : 100
  n_features           : 10
  n_samples_train      : 800
  n_samples_test       : 200


In [2]:
# FASTAPI — exposer le modèle comme une API REST
# En production, les modèles ML sont servis via des APIs

# Créer le fichier main.py
api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np
import joblib
import json

app = FastAPI(title="ML Model API", version="1.0")

# Charger le modèle au démarrage
model = joblib.load("models/random_forest_v1.pkl")
with open("models/metadata_v1.json") as f:
    metadata = json.load(f)

class PredictionRequest(BaseModel):
    features: list[float]

class PredictionResponse(BaseModel):
    prediction: int
    probability: float
    confidence: str
    model_version: str

@app.get("/")
def root():
    return {"status": "ML API is running", "model": metadata["model_name"]}

@app.get("/health")
def health():
    return {"status": "healthy", "accuracy": metadata["accuracy"]}

@app.get("/model-info")
def model_info():
    return metadata

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    try:
        X = np.array(request.features).reshape(1, -1)
        
        prediction = int(model.predict(X)[0])
        proba = float(model.predict_proba(X)[0].max())
        
        if proba > 0.8:
            confidence = "High"
        elif proba > 0.6:
            confidence = "Medium"
        else:
            confidence = "Low"
        
        return PredictionResponse(
            prediction=prediction,
            probability=proba,
            confidence=confidence,
            model_version=metadata["version"]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

with open("main.py", "w") as f:
    f.write(api_code)

print("✅ Fichier main.py créé !")
print("\n=== STRUCTURE DE L'API ===")
print("""
GET  /          → Status de l'API
GET  /health    → Santé + accuracy du modèle
GET  /model-info → Métadonnées complètes
POST /predict   → Faire une prédiction

Exemple de requête POST :
{
    "features": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}

Réponse :
{
    "prediction": 1,
    "probability": 0.85,
    "confidence": "High",
    "model_version": "1.0"
}
""")

print("Pour lancer l'API dans le terminal :")
print("  uvicorn main:app --reload --host 0.0.0.0 --port 8000")
print("\nPour tester :")
print("  http://localhost:8000/docs  → Interface Swagger automatique")

✅ Fichier main.py créé !

=== STRUCTURE DE L'API ===

GET  /          → Status de l'API
GET  /health    → Santé + accuracy du modèle
GET  /model-info → Métadonnées complètes
POST /predict   → Faire une prédiction

Exemple de requête POST :
{
    "features": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}

Réponse :
{
    "prediction": 1,
    "probability": 0.85,
    "confidence": "High",
    "model_version": "1.0"
}

Pour lancer l'API dans le terminal :
  uvicorn main:app --reload --host 0.0.0.0 --port 8000

Pour tester :
  http://localhost:8000/docs  → Interface Swagger automatique


In [2]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import os

# Dataset
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Configurer MLflow avec SQLite
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("RandomForest_Experiment")

configurations = [
    {"n_estimators": 50,  "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": None},
]

print("=== MLFLOW — TRACKING DES EXPÉRIENCES ===\n")

for config in configurations:
    with mlflow.start_run(run_name=f"RF_n{config['n_estimators']}_d{config['max_depth']}"):
        model = RandomForestClassifier(random_state=42, **config)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred)

        mlflow.log_params(config)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.sklearn.log_model(model, "model")

        run_id = mlflow.active_run().info.run_id
        print(f"Config : {config}")
        print(f"  Accuracy : {acc:.4f} | F1 : {f1:.4f}")
        print(f"  Run ID   : {run_id[:8]}...")
        print()

print("✅ Expériences enregistrées !")
print("\nPour voir l'interface MLflow :")
print("  mlflow ui --backend-store-uri sqlite:///mlflow.db")
print("  Puis ouvre : http://localhost:5000")

2026/06/21 09:52:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/21 09:52:07 INFO mlflow.store.db.utils: Updating database tables
2026/06/21 09:52:07 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_Experiment' does not exist. Creating a new experiment.


=== MLFLOW — TRACKING DES EXPÉRIENCES ===



2026/06/21 09:52:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 09:52:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Config : {'n_estimators': 50, 'max_depth': 3}
  Accuracy : 0.8600 | F1 : 0.8704
  Run ID   : 3a298cfc...

Config : {'n_estimators': 100, 'max_depth': 5}
  Accuracy : 0.8650 | F1 : 0.8744
  Run ID   : fccce6b8...



2026/06/21 09:52:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Config : {'n_estimators': 200, 'max_depth': None}
  Accuracy : 0.8800 | F1 : 0.8889
  Run ID   : cdcb81d9...

✅ Expériences enregistrées !

Pour voir l'interface MLflow :
  mlflow ui --backend-store-uri sqlite:///mlflow.db
  Puis ouvre : http://localhost:5000


In [3]:
# DOCKER — Containeriser notre modèle ML
# Docker permet d'empaqueter l'application + ses dépendances
# dans un container portable qui tourne partout

# Créer le Dockerfile
dockerfile = '''
# Image de base Python
FROM python:3.11-slim

# Répertoire de travail
WORKDIR /app

# Copier les dépendances
COPY requirements.txt .

# Installer les dépendances
RUN pip install --no-cache-dir -r requirements.txt

# Copier le code et le modèle
COPY main.py .
COPY models/ models/

# Exposer le port
EXPOSE 8000

# Commande de démarrage
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

# Créer le requirements.txt
requirements = '''fastapi
uvicorn
scikit-learn
joblib
numpy
pydantic
'''

# Créer le docker-compose.yml
docker_compose = '''
version: "3.8"

services:
  ml-api:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - ./models:/app/models
    environment:
      - MODEL_PATH=models/random_forest_v1.pkl
    restart: always
    
  # Service de monitoring (optionnel)
  prometheus:
    image: prom/prometheus
    ports:
      - "9090:9090"
'''

import os
with open("Dockerfile", "w") as f:
    f.write(dockerfile)

with open("requirements.txt", "w") as f:
    f.write(requirements)

with open("docker-compose.yml", "w") as f:
    f.write(docker_compose)

print("✅ Fichiers Docker créés !")
print("\n=== FICHIERS CRÉÉS ===")
for fichier in ["Dockerfile", "requirements.txt", "docker-compose.yml"]:
    print(f"  {fichier}")

print("\n=== COMMANDES DOCKER ESSENTIELLES ===")
print("""
# Construire l'image
docker build -t ml-api:v1 .

# Lancer le container
docker run -p 8000:8000 ml-api:v1

# Lancer avec docker-compose
docker-compose up -d

# Voir les containers en cours
docker ps

# Voir les logs
docker logs <container_id>

# Arrêter
docker-compose down
""")

print("=== POURQUOI DOCKER EN ML ? ===")
print("""
Sans Docker :
  "Ça marche sur ma machine mais pas en prod !"
  
Avec Docker :
  Le container contient Python 3.11, toutes les libs,
  le modèle, et l'API → tourne partout identiquement.
  
En production ML :
  Dev → Container → Test → Container → Prod → Container
  Même environment partout = reproductibilité garantie
""")

✅ Fichiers Docker créés !

=== FICHIERS CRÉÉS ===
  Dockerfile
  requirements.txt
  docker-compose.yml

=== COMMANDES DOCKER ESSENTIELLES ===

# Construire l'image
docker build -t ml-api:v1 .

# Lancer le container
docker run -p 8000:8000 ml-api:v1

# Lancer avec docker-compose
docker-compose up -d

# Voir les containers en cours
docker ps

# Voir les logs
docker logs <container_id>

# Arrêter
docker-compose down

=== POURQUOI DOCKER EN ML ? ===

Sans Docker :
  "Ça marche sur ma machine mais pas en prod !"

Avec Docker :
  Le container contient Python 3.11, toutes les libs,
  le modèle, et l'API → tourne partout identiquement.

En production ML :
  Dev → Container → Test → Container → Prod → Container
  Même environment partout = reproductibilité garantie



In [5]:
# GITHUB ACTIONS — CI/CD pour ML
# Automatiser : tests → entraînement → déploiement

github_actions_yml = '''
name: ML Pipeline CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v3
      
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.11"
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Run tests
        run: pytest tests/ -v
      
      - name: Train model
        run: python train.py
      
      - name: Evaluate model
        run: python evaluate.py
      
      - name: Build Docker image
        run: docker build -t ml-api:latest .
      
      - name: Push to Docker Hub
        if: github.ref == "refs/heads/main"
        run: |
          docker login -u ${{ secrets.DOCKER_USERNAME }} -p ${{ secrets.DOCKER_PASSWORD }}
          docker push ml-api:latest
      
      - name: Deploy to server
        if: github.ref == "refs/heads/main"
        run: |
          ssh user@server "docker pull ml-api:latest && docker-compose up -d"
'''

import os
os.makedirs(".github/workflows", exist_ok=True)
with open(".github/workflows/ml_pipeline.yml", "w") as f:
    f.write(github_actions_yml)

print("✅ GitHub Actions workflow créé !")
print("\n=== FLUX CI/CD COMPLET ===")
print("""
Developer pushes code
        ↓
GitHub Actions déclenché
        ↓
1. Tests unitaires
        ↓
2. Entraîner le modèle
        ↓
3. Évaluer les métriques
        ↓
4. Build Docker image
        ↓
5. Push sur Docker Hub
        ↓
6. Deploy sur le serveur
        ↓
API mise à jour en production !
""")

print("=== MONITORING — métriques à surveiller ===")
print("""
En production, surveiller :
  • Latence API        → temps de réponse < 200ms
  • Data drift         → les inputs changent-ils ?
  • Model performance  → accuracy reste stable ?
  • Erreurs            → taux d'erreur < 1%
  • CPU/RAM/GPU        → ressources utilisées
  
Outils : Prometheus + Grafana, Datadog, MLflow
""")

✅ GitHub Actions workflow créé !

=== FLUX CI/CD COMPLET ===

Developer pushes code
        ↓
GitHub Actions déclenché
        ↓
1. Tests unitaires
        ↓
2. Entraîner le modèle
        ↓
3. Évaluer les métriques
        ↓
4. Build Docker image
        ↓
5. Push sur Docker Hub
        ↓
6. Deploy sur le serveur
        ↓
API mise à jour en production !

=== MONITORING — métriques à surveiller ===

En production, surveiller :
  • Latence API        → temps de réponse < 200ms
  • Data drift         → les inputs changent-ils ?
  • Model performance  → accuracy reste stable ?
  • Erreurs            → taux d'erreur < 1%
  • CPU/RAM/GPU        → ressources utilisées

Outils : Prometheus + Grafana, Datadog, MLflow



In [3]:
import os

# Créer le dossier d'abord
os.makedirs("tests", exist_ok=True)

test_code = '''
import pytest
import numpy as np
import joblib
from fastapi.testclient import TestClient
from main import app

class TestModel:
    def setup_method(self):
        self.model = joblib.load("models/random_forest_v1.pkl")
    
    def test_model_loaded(self):
        assert self.model is not None
    
    def test_prediction_shape(self):
        X = np.random.randn(5, 10)
        predictions = self.model.predict(X)
        assert predictions.shape == (5,)
    
    def test_prediction_binary(self):
        X = np.random.randn(100, 10)
        predictions = self.model.predict(X)
        assert set(predictions).issubset({0, 1})
    
    def test_prediction_proba_sum(self):
        X = np.random.randn(10, 10)
        probas = self.model.predict_proba(X)
        assert np.allclose(probas.sum(axis=1), 1.0)

class TestAPI:
    def test_root_endpoint(self):
        client = TestClient(app)
        response = client.get("/")
        assert response.status_code == 200
    
    def test_health_endpoint(self):
        client = TestClient(app)
        response = client.get("/health")
        assert response.status_code == 200
    
    def test_predict_endpoint(self):
        client = TestClient(app)
        payload = {"features": [0.1] * 10}
        response = client.post("/predict", json=payload)
        assert response.status_code == 200
        data = response.json()
        assert "prediction" in data
        assert data["prediction"] in [0, 1]
        assert 0 <= data["probability"] <= 1

class TestPreprocessing:
    def test_normalisation(self):
        X = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
        X_norm = (X - X.min()) / (X.max() - X.min())
        assert X_norm.min() >= 0
        assert X_norm.max() <= 1
    
    def test_standardisation(self):
        X = np.random.randn(1000) * 5 + 10
        X_std = (X - X.mean()) / X.std()
        assert abs(X_std.mean()) < 0.01
        assert abs(X_std.std() - 1.0) < 0.01
'''

with open("tests/__init__.py", "w") as f:
    f.write("")

with open("tests/test_ml.py", "w") as f:
    f.write(test_code)

print("✅ Fichier de tests créé !")
print("\nPour lancer les tests dans le terminal :")
print("  cd ~/ML_Roadmap/Phase8_MLOps")
print("  pytest tests/ -v")

✅ Fichier de tests créé !

Pour lancer les tests dans le terminal :
  cd ~/ML_Roadmap/Phase8_MLOps
  pytest tests/ -v
